# Upload a resume and view its LangSmith trace

Run the cells in order. Choose a new resume with the upload button, then run extraction and analysis. The dashboard shows stage timing, status, IDs, and counts; resume text and extracted personal details stay out of tracing payloads. The resume itself is processed by the existing OpenAI extraction/analysis calls.

Select the project `.venv` kernel. If widgets are missing, run `uv sync --locked --group dev` in the terminal and restart the kernel. PDF upload requires `pdftotext` (already available in this workspace); scanned PDFs need OCR first.


In [ ]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the InterviewGapAI project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

missing = [key for key in ("LANGSMITH_API_KEY", "OPENAI_API_KEY") if not os.getenv(key)]
if missing:
    raise RuntimeError("Add these keys to the root .env, then rerun setup: " + ", ".join(missing))

from src.observability import configure_observability
from src.observability.notebook import read_uploaded_resume, run_resume_intake, get_trace_url

PROJECT = os.getenv("LANGSMITH_PROJECT") or "interviewgap-ai"
# Explicitly enable export for this notebook; does not edit your .env file.
configure_observability(tracing_enabled=True, project=PROJECT)
print("LangSmith project:", PROJECT)
print("Keys configured. Raw resume text is not captured in traces.")


## Upload one resume

Choose a `.txt`, `.md`, or text-based `.pdf` file. Upload bytes are held in kernel memory, and PDF conversion uses a temporary directory that is removed afterward. Do not save widget state containing the uploaded resume. Clear notebook outputs before sharing.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

upload = widgets.FileUpload(accept=".txt,.md,.pdf", multiple=False, description="Upload resume")
display(upload)


## Run the resume analysis

This cell makes **two live OpenAI calls** using the existing extraction and analysis model (`gpt-5.6`). Set `INCLUDE_INTERVIEW=True` to also generate the interview plan and retrieve questions; this adds a planning call and requires Pinecone access. Each rerun creates a fresh trace and repeats those calls.


In [ ]:
INCLUDE_INTERVIEW = False

if not upload.value:
    raise ValueError("Choose a resume in the upload widget first.")
if INCLUDE_INTERVIEW and not os.getenv("PINECONE_API_KEY"):
    raise RuntimeError("Configure PINECONE_API_KEY for interview question retrieval.")

# Reset old results before a new attempt so the dashboard cell cannot show a stale run.
intake = None
item = upload.value[0]
resume_text = read_uploaded_resume(item["name"], item["content"])
upload.value = ()
# Keep a known interview ID even if a stage fails, to locate the failed run.
from uuid import uuid4
INTERVIEW_ID = "intake-" + uuid4().hex
print("Interview ID:", INTERVIEW_ID)
try:
    intake = run_resume_intake(resume_text, interview_id=INTERVIEW_ID, include_interview=INCLUDE_INTERVIEW)
finally:
    del resume_text, item

print("Trace ID:", intake["trace_id"])
print("Competencies assessed:", len(intake["analysis"].competency_evidence))
if INCLUDE_INTERVIEW:
    print("Questions:", len(intake["question_set"].questions))


## Open the trace in LangSmith

This cell only looks up the dashboard link; it does **not** repeat analysis. If ingestion is still pending, rerun this cell after a few seconds. Sign in to the LangSmith workspace associated with your API key. You can also find the run named `intake.resume` in the project, using its interview ID metadata.


In [ ]:
from IPython.display import HTML
from html import escape

if intake is None:
    print("No successful intake result. Find the failed intake.resume run using the Interview ID printed above.")
else:
    trace_url = get_trace_url(intake["trace_id"])
    if trace_url:
        display(HTML(f'<a href="{escape(trace_url, quote=True)}" target="_blank" rel="noopener noreferrer">Open this resume trace in LangSmith</a>'))
    else:
        print("Trace link not available yet. Rerun this lookup cell; check telemetry.export_failed events if it persists.")
        print("Project:", PROJECT, "Trace ID:", intake["trace_id"])


## Review the analysis locally (optional)

The structured resume and analysis are available as `intake["resume"]` and `intake["analysis"]`. Avoid printing personal details into a notebook that you intend to commit or share. Evidence levels reflect what the resume states, not verified candidate skill.


In [ ]:
if intake is not None:
    for evidence in intake["analysis"].competency_evidence:
        print(evidence.competency.value, evidence.evidence_level.value, evidence.probe_priority.value)
